In [1]:
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path

st.set_page_config(
    page_title='Bone Relapse Predictor',
    page_icon='🧬',
    layout='centered'
)

@st.cache_resource
def cargar_modelo():
    modelo   = joblib.load('mejor_modelo.pkl')
    scaler   = joblib.load('scaler.pkl')
    features = joblib.load('features_list.pkl')
    umbral   = joblib.load('umbral_optimo.pkl')
    return modelo, scaler, features, umbral

modelo, scaler, features, umbral = cargar_modelo()

2026-05-13 12:50:20.948 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-13 12:50:20.954 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-13 12:50:22.859 
  command:

    streamlit run C:\Users\iselo\anaconda3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-05-13 12:50:22.861 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-13 12:50:22.863 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-13 12:50:22.864 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-13 12:50:23.379 Thread 'Thread-3': missing ScriptRunContext! This warning can be ignored when runnin

In [2]:
st.title('🧬 Bone Relapse Risk Predictor')
st.markdown("""
Predicts **bone relapse** risk in breast cancer patients
using gene expression profiles from primary tumors.

**Model:** Logistic Regression — GSE2034 (286 patients)  
**Features:** 35 DEGs | **AUC:** 0.697 | **Threshold:** 0.30
""")

st.divider()

with st.sidebar:
    st.header('About this tool')
    st.markdown("""
    **Dataset:** GSE2034  
    Wang et al. 2005, *The Lancet*
    
    **Pipeline:**  
    1. QC + log2 normalization  
    2. DEG analysis (t-test + FDR)  
    3. PCA + K-means clustering  
    4. Logistic Regression  
    
    **Threshold:** 0.30 (max recall)
    
    ⚠️ Research purposes only.
    """)
    st.divider()
    st.markdown('**Genes used:**')
    for g in features[:10]:
        st.markdown(f'- `{g}`')
    st.markdown(f'... and {len(features)-10} more')

2026-05-13 12:41:14.364 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-13 12:41:14.366 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-13 12:41:14.368 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-13 12:41:14.370 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-13 12:41:14.371 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-13 12:41:14.373 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-13 12:41:14.376 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-13 12:41:14.380 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [6]:
st.header('Upload Gene Expression Data')

col1, col2 = st.columns([2, 1])
with col1:
    archivo = st.file_uploader(
        'Upload CSV file (patients as rows, genes as columns)',
        type=['csv']
    )
with col2:
    st.markdown('**Expected format:**')
    st.markdown('- Rows: patients')
    st.markdown('- Columns: gene IDs')
    st.markdown('- Values: log2 expression')

if archivo is not None:
    try:
        df = pd.read_csv(archivo, index_col=0)
        st.success(f'File loaded: {df.shape[0]} patients, {df.shape[1]} genes')
        
        genes_encontrados = [g for g in features if g in df.columns]
        genes_faltantes   = [g for g in features if g not in df.columns]
        
        if len(genes_encontrados) < 10:
            st.error(f'Only {len(genes_encontrados)} of {len(features)} required genes found. '
                     f'Check that gene IDs match Affymetrix HG-U133A format.')
            st.stop()
        
        if genes_faltantes:
            st.warning(f'{len(genes_faltantes)} genes not found — '
                       f'filling with mean expression values.')
            for g in genes_faltantes:
                df[g] = df[genes_encontrados].mean(axis=1)
        
        X_pred = df[features].values
        X_pred_sc = scaler.transform(X_pred)
        probabilidades = modelo.predict_proba(X_pred_sc)[:, 1]
        predicciones   = (probabilidades >= umbral).astype(int)
        
        st.divider()
        st.header('Prediction Results')
        
        df_result = pd.DataFrame({
            'Patient':           df.index,
            'Relapse Probability': probabilidades.round(3),
            'Risk Level':        ['HIGH' if p >= umbral else 'LOW' for p in probabilidades],
            'Prediction':        ['Bone relapse risk' if p == 1 else 'Low risk' for p in predicciones]
        })
        
        col_a, col_b, col_c = st.columns(3)
        col_a.metric('Patients analyzed', len(df_result))
        col_b.metric('High risk', int(predicciones.sum()),
                     delta=f'{predicciones.mean()*100:.0f}%')
        col_c.metric('Low risk', int((predicciones==0).sum()))
        
        st.dataframe(
            df_result.style.applymap(
                lambda v: 'background-color: #FFE5E5; color: #8B0000' if v == 'HIGH'
                          else 'background-color: #E5FFE5; color: #006400',
                subset=['Risk Level']
            ),
            use_container_width=True
        )
        
    except Exception as e:
        st.error(f'Error processing file: {str(e)}')
else:
    st.info('👆 Upload a CSV file to get predictions')
    st.markdown('**No data? Use the sample file:**')
    if Path('muestra_test.csv').exists():
        with open('muestra_test.csv', 'rb') as f:
            st.download_button(
                label='⬇️ Download sample file',
                data=f,
                file_name='muestra_test.csv',
                mime='text/csv'
            )
        st.divider()
        st.subheader('Risk Probability Distribution')
        
        fig = go.Figure()
        
        colores_bar = ['#E84C4C' if p >= umbral else '#4C9BE8' for p in probs]
        
        fig.add_trace(go.Bar(
            x=list(df_res['Patient']),
            y=list(probs),
            marker_color=colores_bar,
            text=[f'{p:.2f}' for p in probs],
            textposition='outside'
        ))
        
        fig.add_hline(
            y=umbral,
            line_dash='dash',
            line_color='black',
            annotation_text=f'Risk threshold ({umbral})',
            annotation_position='top right'
        )
        
        fig.update_layout(
            title='Bone Relapse Probability per Patient',
            xaxis_title='Patient ID',
            yaxis_title='Probability of Bone Relapse',
            yaxis=dict(range=[0, 1.1]),
            showlegend=False,
            height=400
        )
        
        st.plotly_chart(fig, use_container_width=True)
        
        csv_result = df_res.to_csv(index=False)
        st.download_button(
            label='⬇️ Download results as CSV',
            data=csv_result,
            file_name='bone_relapse_predictions.csv',
            mime='text/csv'
        )

2026-05-13 12:42:52.901 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-13 12:42:52.902 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-13 12:42:52.904 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-13 12:42:52.908 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-13 12:42:52.910 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-13 12:42:52.914 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-13 12:42:52.917 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-13 12:42:52.920 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar